# Task 2: Record the Idle Class (and optional calibration clips)

**The 25 signs are trained from the public ASL dataset** (see README, "Training
data") — you do **not** need to record them.

This notebook exists because the dataset has **no `Idle` class** (natural,
non-signing hand movement). Without it, the live system must guess one of the
25 words whenever your hand moves. By default this notebook records only `Idle`.

Optional: set `SIGNS = ALL_SIGNS + ["Idle"]` in the config cell to also record
a few of your own clips per sign. They're only used to measure how well the
dataset-trained model transfers to your webcam (domain-gap cell in notebook 03).

**Protocol:**
- Show only one hand in frame.
- Press **r** to start recording a clip — it auto-stops after a fixed duration.
- For `Idle`, move your hand around naturally (rest it, reposition it, scratch your head, start to raise it and stop) — anything that is *not* one of the signs.
- If a clip gets discarded (hand lost too much), just press **r** again to retry.

Run cells top to bottom. Kernel: **Python (ipcv-task2)**.

## 1. Config — edit this to change the sign vocabulary or who's collecting

In [ ]:
# The 25 dataset words (trained from the public dataset, NOT recorded here).
ALL_SIGNS = [
    "Hello", "SeeYouLater", "IMe", "Father", "Mother", "Yes", "No", "Help",
    "Please", "ThankYou", "Want", "What", "Dog", "Cat", "Again", "EatFood",
    "Milk", "More", "GoTo", "Bathroom", "Fine", "Like", "Learn", "Sign", "Finish",
]

# DEFAULT: record only Idle (it is not in the dataset, so it must come from you).
SIGNS = ["Idle"]
# The vocabulary currently trained (see notebook 03, KEEP_WORDS):
KEPT_WORDS = ["Apple", "Please", "Color", "Yes", "No", "More", "Boy", "Hello", "Bathroom", "Yesterday", "Brown", "Football", "Water", "Red"]
# Optional: record your OWN clips of the kept words (best way to make them reliable on your webcam):
# SIGNS = KEPT_WORDS + ["Idle"]   # and raise CLIPS_PER_SIGN (e.g. 25)

# Change this each time a different team member collects data.
COLLECTOR = "student1"

CLIPS_PER_SIGN = 10            # per real sign, only used if you add signs to SIGNS
CLIPS_OVERRIDE = {"Idle": 40}  # Idle clips needed (it is NOT in the dataset)
CLIP_DURATION = 1.2            # seconds per recorded clip
TARGET_FRAMES = 20             # clip is resampled to this many evenly spaced time steps

print(f"Recording: {SIGNS}")
print(f"Total clips target: {sum(CLIPS_OVERRIDE.get(s, CLIPS_PER_SIGN) for s in SIGNS)}")
print(f"Each clip: {CLIP_DURATION}s, resampled to {TARGET_FRAMES} frames.")

## 2. Imports and model loading

Uses the shared `src/features.py` and `src/hand_model.py` so data collection,
training, and live recognition all use identical math.

In [ ]:
import sys, os, importlib
sys.path.append(os.path.abspath(".."))

import time
from datetime import datetime

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

import src.features, src.hand_model
# Pick up edits to src/ without restarting the kernel (a kernel keeps the
# first-imported copy of a module in memory otherwise).
importlib.reload(src.features)
importlib.reload(src.hand_model)

from src.features import extract_frame, build_clip_feature
from src.hand_model import load_hand_landmarker

landmarker = load_hand_landmarker(model_path="../models/hand_landmarker.task", num_hands=1)
print("HandLandmarker loaded OK")

## 3. Recording loop

Controls (video window must be focused):
- **r** — start recording a clip of the current sign (auto-stops after `CLIP_DURATION` seconds)
- **n** — skip to the next sign early (only works when not currently recording)
- **q** — quit and save everything collected so far

A red progress bar shows how far through the current clip you are.

In [ ]:
FONT = cv2.FONT_HERSHEY_SIMPLEX

records = []  # list of dicts: {feature_0..(TARGET_FRAMES*63-1), label, collector, timestamp}
sign_index = 0
clip_count = 0

recording = False
record_start_time = None
clip_frames, clip_scales, clip_times = [], [], []

cap = cv2.VideoCapture(0)
print(f">>> Sign: {SIGNS[0]}  — press 'r' to record a clip")

try:
    while sign_index < len(SIGNS):
        current_sign = SIGNS[sign_index]
        target_clips = CLIPS_OVERRIDE.get(current_sign, CLIPS_PER_SIGN)

        ret, frame = cap.read()
        if not ret:
            print("Frame grab failed, stopping.")
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)
        hand_visible = len(result.hand_landmarks) > 0

        now = time.time()
        elapsed = (now - record_start_time) if recording else 0.0

        if recording:
            if hand_visible:
                raw63, hand_scale = extract_frame(result.hand_landmarks[0], aspect=frame.shape[1] / frame.shape[0])
                clip_frames.append(raw63)
                clip_scales.append(hand_scale)
                clip_times.append(elapsed)

            if elapsed >= CLIP_DURATION:
                seq = (
                    build_clip_feature(clip_frames, clip_scales, clip_times, TARGET_FRAMES)
                    if len(clip_frames) >= 5 else None
                )
                if seq is not None:
                    record = {f"feature_{i}": v for i, v in enumerate(seq)}
                    record["label"] = current_sign
                    record["collector"] = COLLECTOR
                    record["timestamp"] = datetime.now().isoformat()
                    records.append(record)
                    clip_count += 1
                    print(f"  captured clip {clip_count}/{target_clips} for '{current_sign}'")
                else:
                    print("  clip discarded (hand not detected enough) — press 'r' to retry")

                recording = False
                clip_frames, clip_scales, clip_times = [], [], []

                if clip_count >= target_clips:
                    sign_index += 1
                    clip_count = 0
                    if sign_index < len(SIGNS):
                        print(f">>> Sign: {SIGNS[sign_index]}  — press 'r' to record a clip")

        # --- overlay ---
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 110), (40, 40, 40), -1)
        cv2.putText(frame, f"SIGN: {current_sign}", (20, 40), FONT, 1.1, (0, 255, 255), 3)
        status = "RECORDING..." if recording else "press 'r' to record"
        color = (0, 0, 255) if recording else (255, 255, 255)
        cv2.putText(frame, f"{status}   clips: {clip_count}/{target_clips}   (n=next sign  q=quit)",
                    (20, 80), FONT, 0.6, color, 2)
        if recording:
            pct = min(elapsed / CLIP_DURATION, 1.0)
            bar_w = int(300 * pct)
            cv2.rectangle(frame, (20, 95), (320, 105), (100, 100, 100), 1)
            cv2.rectangle(frame, (20, 95), (20 + bar_w, 105), (0, 0, 255), -1)
        if not hand_visible and not recording:
            cv2.putText(frame, "No hand detected", (20, 140), FONT, 0.7, (0, 0, 255), 2)

        cv2.imshow("Task 2 - Dynamic Sign Data Collection", frame)
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break
        elif key == ord("n") and not recording:
            print(f"Skipped rest of '{current_sign}' with {clip_count} clips.")
            sign_index += 1
            clip_count = 0
            if sign_index < len(SIGNS):
                print(f">>> Sign: {SIGNS[sign_index]}  — press 'r' to record a clip")
        elif key == ord("r") and not recording:
            recording = True
            record_start_time = now
            clip_frames, clip_scales, clip_times = [], [], []
finally:
    cap.release()
    cv2.waitKey(1)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

print(f"\nTotal clips collected this session: {len(records)}")

## 4. Save to CSV

Saved under `../data/raw/` (gitignored — each teammate keeps their own raw
clips locally; `03_train_classifier.ipynb` merges all CSVs found there).

In [ ]:
if records:
    df = pd.DataFrame(records)
    out_path = f"../data/raw/{COLLECTOR}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved {len(df)} clips to {out_path}")
    print(df["label"].value_counts())
else:
    print("No records collected — nothing to save.")

## Cleanup

In [ ]:
landmarker.close()
print("Landmarker closed.")

## Next steps

1. Run `03_train_classifier.ipynb` — it trains on the dataset clips
   (`data/raw/hf_dataset_*.csv`) plus the Idle clips you just recorded.
2. Test live with `04_live_recognition.ipynb`.
3. Optional: record a few of your own clips per sign (see top of this notebook)
   to measure the dataset-to-webcam accuracy gap.